In [ ]:
import os
import json
import time
from llama_cpp import Llama

# ================== Configuration ==================
MODEL_PATH = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/DeepSeek_R1_Distill_Qwen_14B_output_q8.gguf"  # Replace with your GGUF path
INPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/TiJian94.json"           # 219 records
OUTPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/results_with_deepseek.json"
GPU_LAYERS = -1
N_CTX = 32768               
N_BATCH = 1024
N_UBATCH = 512
N_THREADS = 4
MAIN_GPU = 0
MAX_TOKENS = 4096
TEMPERATURE = 0.1
TOP_P = 0.8
# Stop tokens: The training data has no special end marker, so the model's native eos_token can be used; common ones are listed here
STOP_TOKENS = ["<|im_end|>", "<|endoftext|>", "###"]   # "###" might cause premature truncation, keep or remove based on actual conditions
# ========================================

# 1. Load model
print("Loading model...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_batch=N_BATCH,
    n_ubatch=N_UBATCH,
    n_threads=N_THREADS,
    n_gpu_layers=GPU_LAYERS,
    main_gpu=MAIN_GPU,
    chat_format=None,               # Do not use any predefined chat format
    flash_attn=True,
    f16_kv=True,
    use_mmap=True,
    use_mlock=False,
    offload_kqv=True,
    verbose=False,
)
print("Model loaded.")

# 2. Read data
with open(INPUT_JSON, 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f"Total records: {len(data)}")

# 3. Batch processing
results = []
for idx, item in enumerate(data):
    instruction = item.get("instruction", "")
    input_text = item.get("input", "")

    # Construct a prompt exactly consistent with training (Alpaca style)
    # Note: During training, the output directly follows "### 回答:\n", so this prefix is needed here to guide generation
    if input_text:
        prompt = f"### 指令:\n{instruction}\n\n### 输入:\n{input_text}\n\n### 回答:\n"
    else:
        prompt = f"### 指令:\n{instruction}\n\n### 回答:\n"

    print(f"\nProcessing {idx+1}/{len(data)} ...")

    try:
        response = llm.create_completion(
            prompt=prompt,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
            stop=STOP_TOKENS,
            echo=False,            # Do not return the prompt part
            stream=False,
        )
        generated_text = response["choices"][0]["text"]
    except Exception as e:
        print(f"Error on item {idx+1}: {e}")
        print(f"Failed item: instruction={instruction[:100]}..., input={input_text[:100]}...")  # Print partial content for easier debugging
        generated_text = ""

    # Keep original fields, add output_DeepSeek
    new_item = item.copy()
    new_item["output_DeepSeek"] = generated_text
    results.append(new_item)

    # Save intermediate results every 10 records
    if (idx + 1) % 10 == 0:
        with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"Checkpoint saved at {idx+1} records.")

# 4. Final save
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nAll done! Results saved to {OUTPUT_JSON}")